In [1]:
# R5_venn_abers.ipynb  (revision stage 5 - calibrated PD intervals, completes M5)
# Binary split-conformal cannot produce a probability-of-default (PD) interval: the
# event "this applicant defaults" is observed 0/1, not as a probability, so a
# conformal set on the label says nothing about an interval for P(default).  The
# proper tool (Vovk & Petej 2014) is a Venn-Abers predictor, which outputs for each
# applicant a calibrated multiprobability interval [p0, p1] for the PD, automatically
# calibrated and with width that widens where evidence is thin.
# Here we fit an Inductive Venn-Abers Predictor (IVAP, isotonic-regression based,
# implemented self-contained) on top of BOTH interpretable models on the same WoE
# grid, and ask the second-pillar question in its correct form: do the logistic-free
# stump booster and logistic-WoE deliver PD intervals of the same width and
# calibration?  Binning + IV INSIDE the training fold (M6). Single cell; greyscale;
# dpi600 png+pdf.

import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')

SEED=42; N_REP=10; IV_THRESH=0.02
N_CAL=3000; N_TEST=1500            # subsampled for IVAP cost; evaluation is unbiased
sns.set_theme(style='whitegrid', context='paper'); sns.set_palette('Greys')
plt.rcParams.update({'axes.edgecolor':'0.2','grid.color':'0.85','font.size':10})
ROOT=os.path.abspath(os.path.join(os.getcwd(),'..')); DATA=os.path.join(ROOT,'data')
TABLES=os.path.join(ROOT,'results','tables'); FIGS=os.path.join(ROOT,'results','figures')
def savefig(fig,name):
    for e in ('png','pdf'): fig.savefig(os.path.join(FIGS,name+'.'+e),dpi=600,bbox_inches='tight')
    plt.close(fig)

dfc=pd.read_parquet(os.path.join(DATA,'gmsc_clean.parquet'))
TARGET='SeriousDlqin2yrs'; FEATURES=[c for c in dfc.columns if c!=TARGET]
y_all=dfc[TARGET].values.astype(int); X_all=dfc[FEATURES]
print('GMSC clean:',dfc.shape)

# ---- in-fold binning (identical to R1/R2/R3) ----
def tree_edges(x,yv,max_leaf=6,min_frac=0.05):
    m=x.notna().values; xx=x[m].values.reshape(-1,1); yy=yv[m]
    if len(np.unique(xx))<=2: return []
    t=DecisionTreeClassifier(max_leaf_nodes=max_leaf,min_samples_leaf=int(min_frac*len(xx)),random_state=SEED)
    t.fit(xx,yy); thr=t.tree_.threshold[t.tree_.feature>=0]; return sorted(float(v) for v in thr)
def bin_counts(x,yv,edges,TG,TB,min_n):
    cuts=[-np.inf]+list(edges)+[np.inf]; idx=pd.cut(x,cuts,labels=False,include_lowest=True); recs=[]
    for b in range(len(cuts)-1):
        sel=idx==b; g=int(((yv==0)&sel).sum()); bd=int(((yv==1)&sel).sum())
        if g+bd>0: recs.append({'lo':cuts[b],'hi':cuts[b+1],'good':g,'bad':bd,'miss':False})
    selm=x.isna(); gm=int(((yv==0)&selm).sum()); bm=int(((yv==1)&selm).sum())
    if gm+bm>0: recs.append({'lo':np.nan,'hi':np.nan,'good':gm,'bad':bm,'miss':True})
    for r in recs:
        gr=(r['good']+0.5)/(TG+0.5); br=(r['bad']+0.5)/(TB+0.5); r['woe']=float(np.log(br/gr)); r['n']=r['good']+r['bad']
        if r['miss'] and r['n']<min_n: r['woe']=0.0
    return recs
def enforce_mono(x,yv,edges,TG,TB,min_n):
    e=list(edges)
    while e:
        recs=[r for r in bin_counts(x,yv,e,TG,TB,min_n) if not r['miss']]; br=np.array([r['bad']/r['n'] for r in recs])
        if len(br)<=2 or np.all(np.diff(br)>=-1e-12) or np.all(np.diff(br)<=1e-12): return e
        d=np.diff(br); s=np.sign(np.nansum(d)); e.pop(int(np.argmax(-s*d)))
    return e
def fit_binning(Xtr,ytr):
    TG=int((ytr==0).sum()); TB=int((ytr==1).sum()); min_n=int(0.01*len(ytr)); maps={}; iv={}
    for c in FEATURES:
        recs=bin_counts(Xtr[c],ytr,enforce_mono(Xtr[c],ytr,tree_edges(Xtr[c],ytr),TG,TB,min_n),TG,TB,min_n); maps[c]=recs
        iv[c]=float(np.sum([((r['bad']+.5)/(TB+.5)-(r['good']+.5)/(TG+.5))*r['woe'] for r in recs]))
    return maps,[c for c in FEATURES if iv[c]>=IV_THRESH]
def apply_bins(x,recs):
    idx=np.full(len(x),-1,int); fin=[r for r in recs if not r['miss']]
    for p,r in enumerate(fin):
        sel=(~x.isna()).values&(x>r['lo']).values&(x<=r['hi']).values; idx[sel]=p
    mp=[p for p,r in enumerate(recs) if r['miss']]; idx[x.isna().values]=mp[0] if mp else 0; idx[idx==-1]=0; return idx
def woe_mat(Xdf,maps,keep):
    return np.column_stack([np.array([r['woe'] for r in maps[c]])[apply_bins(Xdf[c],maps[c])] for c in keep])

# ---- Inductive Venn-Abers Predictor (isotonic, exact per test point) ----
def ivap(cal_s, cal_y, test_s):
    cal_s=np.asarray(cal_s,float); cal_y=np.asarray(cal_y,float); test_s=np.asarray(test_s,float)
    p0=np.empty(len(test_s)); p1=np.empty(len(test_s))
    base_x=cal_s
    for i,s in enumerate(test_s):
        xs=np.append(base_x,s)
        ir0=IsotonicRegression(y_min=0,y_max=1,increasing=True,out_of_bounds='clip').fit(xs,np.append(cal_y,0.0))
        ir1=IsotonicRegression(y_min=0,y_max=1,increasing=True,out_of_bounds='clip').fit(xs,np.append(cal_y,1.0))
        p0[i]=ir0.predict([s])[0]; p1[i]=ir1.predict([s])[0]
    return p0,p1
# sanity: p0<=p1 and VA prob in [0,1]
_c=np.linspace(0,1,200); _y=(np.random.default_rng(0).random(200)<_c).astype(float)
_p0,_p1=ivap(_c,_y,np.array([0.2,0.8])); assert np.all(_p0<=_p1+1e-9), 'IVAP p0<=p1 violated'

def brier(p,y): return float(np.mean((p-y)**2))
def ece(p,y,bins=10):
    e=0.0; edges=np.linspace(0,1,bins+1)
    for i in range(bins):
        m=(p>=edges[i])&(p<edges[i+1] if i<bins-1 else p<=edges[i+1])
        if m.sum()>0: e+=abs(p[m].mean()-y[m].mean())*m.sum()
    return float(e/len(p))

rows=[]; width_scatter={'logit':[], 'stump':[]}; t0=time.time()
rng=np.random.default_rng(SEED)
for rep in range(N_REP):
    Xtr,Xtmp,ytr,ytmp=train_test_split(X_all,y_all,test_size=0.4,stratify=y_all,random_state=SEED+rep)
    Xcal,Xte,ycal,yte=train_test_split(Xtmp,ytmp,test_size=0.5,stratify=ytmp,random_state=SEED+rep)
    # subsample cal/test for IVAP cost (stratified)
    def sub(Xd,yd,n,rs):
        if len(yd)<=n: return Xd,yd
        idx=np.arange(len(yd)); tr,_=train_test_split(idx,train_size=n,stratify=yd,random_state=rs); return Xd.iloc[tr],yd[tr]
    Xcal,ycal=sub(Xcal,ycal,N_CAL,SEED+100+rep); Xte,yte=sub(Xte,yte,N_TEST,SEED+200+rep)
    maps,keep=fit_binning(Xtr,ytr)
    Wtr=woe_mat(Xtr,maps,keep); Wcal=woe_mat(Xcal,maps,keep); Wte=woe_mat(Xte,maps,keep)
    for kind in ('logit','stump'):
        if kind=='logit': m=LogisticRegression(max_iter=1000).fit(Wtr,ytr)
        else: m=GradientBoostingClassifier(max_depth=1,learning_rate=0.3,n_estimators=300,random_state=SEED+rep).fit(Wtr,ytr)
        sc=m.predict_proba(Wcal)[:,1]; st=m.predict_proba(Wte)[:,1]
        p0,p1=ivap(sc,ycal,st)
        pva=p1/(1.0-p0+p1)              # Venn-Abers calibrated point probability
        width=p1-p0
        rows.append([rep,kind,float(np.median(width)),float(width.mean()),
                     float(np.quantile(width,0.9)),brier(pva,yte),ece(pva,yte),
                     brier(st,yte)])    # uncalibrated model Brier for reference
        if rep==0:
            width_scatter[kind]=np.column_stack([pva,width])
    if (rep+1)%2==0: print('  rep %d/%d (%.0fs)'%(rep+1,N_REP,time.time()-t0))
print('done %.0fs'%(time.time()-t0))

va=pd.DataFrame(rows,columns=['rep','model','width_median','width_mean','width_p90',
                              'brier_va','ece_va','brier_raw'])
summ=va.groupby('model').agg(width_median=('width_median','mean'),width_mean=('width_mean','mean'),
     width_p90=('width_p90','mean'),brier_va=('brier_va','mean'),ece_va=('ece_va','mean'),
     brier_raw=('brier_raw','mean')).round(4)
print('\n=== Venn-Abers PD intervals (GMSC, %d reps) ==='%N_REP)
print(summ.to_string())
# paired difference in median width (stump - logit)
piv=va.pivot(index='rep',columns='model',values='width_median')
dw=(piv['stump']-piv['logit'])
from scipy import stats
tt=stats.ttest_rel(piv['stump'],piv['logit'])
print('\nmedian-width stump-logit: mean diff %.4f  (paired t p=%.3f)'%(dw.mean(),tt.pvalue))
print('width ratio stump/logit: %.3f'%(piv["stump"].mean()/piv["logit"].mean()))

va.round(6).to_csv(os.path.join(TABLES,'R5_va_long.csv'),index=False)
summ.to_csv(os.path.join(TABLES,'R5_va_summary.csv'))

# ---- figures ----
# (1) interval-width distribution, logit vs stump
fig,ax=plt.subplots(figsize=(7.2,4.0))
order=['logit','stump']
sns.boxplot(data=va.assign(model=pd.Categorical(va['model'],order)),x='model',y='width_median',
            ax=ax,width=0.5,fliersize=2,color='0.7')
ax.set_xticklabels(['logistic-WoE','stump boosting']); ax.set_xlabel('')
ax.set_ylabel('median Venn-Abers PD-interval width')
savefig(fig,'R5_va_width')

# (2) width vs predicted PD (rep 0), both models
fig,ax=plt.subplots(figsize=(7.2,4.0))
for kind,mk,cl in (('logit','o','0.15'),('stump','^','0.55')):
    arr=width_scatter[kind]; o=np.argsort(arr[:,0])
    ax.scatter(arr[o,0],arr[o,1],s=6,marker=mk,color=cl,alpha=0.5,
               label=('logistic-WoE' if kind=='logit' else 'stump boosting'))
ax.set_xlabel('Venn-Abers PD (calibrated $p$)'); ax.set_ylabel('PD-interval width $p_1-p_0$')
ax.set_xscale('log'); ax.legend(frameon=False,loc='upper right')
savefig(fig,'R5_va_width_vs_pd')
print('\nfigures: R5_va_width, R5_va_width_vs_pd  | tables: R5_va_long, R5_va_summary')


GMSC clean: (150000, 11)


  rep 2/10 (22s)


  rep 4/10 (44s)


  rep 6/10 (67s)


  rep 8/10 (89s)


  rep 10/10 (110s)
done 110s

=== Venn-Abers PD intervals (GMSC, 10 reps) ===
       width_median  width_mean  width_p90  brier_va  ece_va  brier_raw
model                                                                  
logit        0.0035      0.0072     0.0139    0.0514  0.0110     0.0515
stump        0.0041      0.0075     0.0153    0.0514  0.0119     0.0514

median-width stump-logit: mean diff 0.0006  (paired t p=0.072)
width ratio stump/logit: 1.168



figures: R5_va_width, R5_va_width_vs_pd  | tables: R5_va_long, R5_va_summary
